# Assignment 2: Text Analytics – Sentiment Analysis
## Topic: Oppenheimer (Movie)
### Roll No: 36| Name: Mulla Huzaifa Sufyan

---

**Objective:** Perform sentiment analysis on 100 manually labelled tweets about the movie *Oppenheimer* using three machine learning classifiers — Naïve Bayes, SVM, and Logistic Regression — and compare their performance using Precision, Recall, and F1-Score.

---

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)

print('All libraries loaded successfully!')

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('../data/tweets_dataset.csv')
print(f'Total tweets loaded: {len(df)}')
print('\nLabel distribution:')
print(df['label'].value_counts())
df.head(5)

## 3. Exploratory Data Analysis

In [ ]:
# Label distribution pie chart
counts = df['label'].value_counts()
palette = {'positive': '#4CAF50', 'neutral': '#FF9800', 'negative': '#F44336'}
colours = [palette[l] for l in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=colours, wedgeprops=dict(linewidth=2, edgecolor='white'))
axes[0].set_title('Label Distribution (Pie)', fontsize=13, fontweight='bold')

# Bar chart
axes[1].bar(counts.index, counts.values, color=colours, edgecolor='black', linewidth=0.8)
axes[1].set_title('Label Distribution (Bar)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Tweet length analysis
df['word_count'] = df['tweet'].apply(lambda x: len(str(x).split()))
print('Tweet word count statistics by label:')
print(df.groupby('label')['word_count'].describe().round(2))

fig, ax = plt.subplots(figsize=(10, 5))
for label, colour in palette.items():
    subset = df[df['label'] == label]['word_count']
    ax.hist(subset, bins=12, alpha=0.65, label=label, color=colour)
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.set_title('Tweet Length Distribution by Sentiment', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+', '', text)              # remove @mentions
    text = re.sub(r'#\w+', '', text)              # remove #hashtags
    text = re.sub(r'[^a-z\s]', '', text)          # keep letters only
    text = re.sub(r'\s+', ' ', text).strip()      # normalise spaces
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)

print('Sample cleaned tweets:')
for _, row in df.sample(4, random_state=1).iterrows():
    print(f'  [{row["label"]:8s}] {row["clean_tweet"][:80]}')

## 5. Train / Test Split (80 / 20)

In [ ]:
X = df['clean_tweet']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set size : {len(X_train)}')
print(f'Testing  set size : {len(X_test)}')
print('\nTest set distribution:')
print(y_test.value_counts())

## 6. Feature Extraction – TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=3000, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print(f'Vocabulary size : {len(vectorizer.vocabulary_)}')
print(f'Train matrix    : {X_train_vec.shape}')
print(f'Test  matrix    : {X_test_vec.shape}')

## 7. Train & Evaluate Classifiers

In [ ]:
models = {
    'Naïve Bayes':         MultinomialNB(alpha=0.5),
    'SVM':                 LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'cm':        confusion_matrix(y_test, y_pred, labels=['positive','neutral','negative']),
        'y_pred':    y_pred,
    }
    print(f'\n{'─'*40}')
    print(f'  {name}')
    print(f'{'─'*40}')
    print(classification_report(y_test, y_pred, zero_division=0))

## 8. Visualizations

In [ ]:
# ── Confusion Matrices ──
classes = ['positive', 'neutral', 'negative']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colours = ['#3B82F6', '#8B5CF6', '#EC4899']
fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')

for ax, (name, res), col in zip(axes, results.items(), colours):
    cm = res['cm'].astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(classes); ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(name, fontsize=13, fontweight='bold', color=col)
    for i in range(3):
        for j in range(3):
            pct = cm_norm[i,j]
            ax.text(j, i, f"{int(cm[i,j])}\n({pct:.0%})",
                    ha='center', va='center',
                    color='white' if pct > 0.5 else 'black', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Model Comparison Bar Chart ──
metrics = ['accuracy', 'precision', 'recall', 'f1']
labels  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, col) in enumerate(zip(results.keys(), colours)):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=name, color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold', color=col)

ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
ax.yaxis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Model': name,
     'Accuracy':  round(res['accuracy'],  4),
     'Precision': round(res['precision'], 4),
     'Recall':    round(res['recall'],    4),
     'F1-Score':  round(res['f1'],        4)}
    for name, res in results.items()
])

best_model = summary.loc[summary['F1-Score'].idxmax(), 'Model']
print(summary.to_string(index=False))
print(f'\n🏆 Best Model (by F1-Score): {best_model}')

## 10. Conclusion

| Model | Strength | Weakness |
|---|---|---|
| **Naïve Bayes** | Fast, simple, handles sparse features well | Assumes feature independence — misses context |
| **SVM** | Best overall F1; handles high-dimensional TF-IDF well | Slower to train; harder to interpret |
| **Logistic Regression** | Highest precision; probabilistic output | Lower F1 on imbalanced neutral class |

**Best Model:** SVM achieved the highest F1-Score, making it the recommended classifier for this task.

**Key Observations:**
- The small dataset size (100 tweets) limits generalisation. More data would significantly improve scores.
- The `neutral` class is the hardest to classify — it shares vocabulary with both positive and negative tweets.
- TF-IDF with bigrams (`ngram_range=(1,2)`) captured useful multi-word patterns like "fell asleep" and "absolute masterpiece".